# IA – Aula 3
## Sistema Baseado em Regras – Classificação de Conforto Térmico

**Objetivo:**
- Implementar um sistema simples de regras.
- Classificar conforto térmico com base em temperatura e umidade.
- Comparar abordagem simbólica (regras) com Machine Learning.


In [1]:
import pandas as pd
from dataclasses import dataclass
from typing import Callable, Dict, List

## Definição das Regras

Regras simplificadas baseadas no diagrama fornecido:
- **Muito Frio:** temperatura < 10°C
- **Muito Quente:** temperatura > 35°C
- **Muito Seco:** umidade < 30%
- **Muito Úmido:** umidade > 80%
- **Confortável:** região intermediária (fallback)

In [2]:
@dataclass
class Rule:
    name: str
    condition: Callable[[Dict], bool]
    label: str


def classify(row, rules: List[Rule]):
    """Percorre as regras em ordem de prioridade.
    Retorna o label da primeira regra que disparar;
    se nenhuma disparar, retorna 'Confortável'."""
    for r in rules:
        if r.condition(row):
            return r.label
    return 'Confortável'

In [3]:
rules = [
    Rule('Muito Frio',   lambda r: r['temperatura'] < 10,  'Muito Frio'),
    Rule('Muito Quente', lambda r: r['temperatura'] > 35,  'Muito Quente'),
    Rule('Muito Seco',   lambda r: r['umidade'] < 30,      'Muito Seco'),
    Rule('Muito Úmido',  lambda r: r['umidade'] > 80,      'Muito Úmido'),
]

## Dataset de Teste

Além dos 5 casos claros, incluímos:
- **Bordas exatas** (temp = 10, 35; umidade = 30, 80) — os limiares usam `<` e `>` estritos, então o valor exato **não** dispara a regra e cai em Confortável.
- **Conflitos de prioridade** (temp < 10 **e** umidade < 30) — a ordem das regras determina qual classificação prevalece.

In [4]:
data = pd.DataFrame({
    'temperatura': [5, 22, 40, 28, 18, 10, 35, 22, 22, 10, 35, 9, 36],
    'umidade':     [50, 45, 60, 85, 20, 50, 50, 30, 80, 30, 80, 25, 90],
    'caso':        [
        'Muito Frio claro',
        'Confortável claro',
        'Muito Quente claro',
        'Muito Úmido claro',
        'Muito Seco claro',
        'Borda: temp = 10 (não dispara Muito Frio)',
        'Borda: temp = 35 (não dispara Muito Quente)',
        'Borda: umidade = 30 (não dispara Muito Seco)',
        'Borda: umidade = 80 (não dispara Muito Úmido)',
        'Borda: temp = 10 e umidade = 30',
        'Borda: temp = 35 e umidade = 80',
        'Conflito: temp < 10 e umidade < 30',
        'Conflito: temp > 35 e umidade > 80',
    ]
})

data['classificacao'] = data.apply(lambda row: classify(row, rules), axis=1)
data[['caso', 'temperatura', 'umidade', 'classificacao']]

,caso,temperatura,umidade,classificacao
0,Muito Frio claro,5,50,Muito Frio
1,Confortável claro,22,45,Confortável
2,Muito Quente claro,40,60,Muito Quente
3,Muito Úmido claro,28,85,Muito Úmido
4,Muito Seco claro,18,20,Muito Seco
5,Borda: temp = 10 (não dispara Muito Frio),10,50,Confortável
6,Borda: temp = 35 (não dispara Muito Quente),35,50,Confortável
7,Borda: umidade = 30 (não dispara Muito Seco),22,30,Confortável
8,Borda: umidade = 80 (não dispara Muito Úmido),22,80,Confortável
9,Borda: temp = 10 e umidade = 30,10,30,Confortável


## Discussão

| Aspecto | Sistema de Regras | Machine Learning |
|---|---|---|
| **Transparência** | Alta – cada decisão é rastreável até a regra que disparou | Baixa a média – modelos como redes neurais são "caixas-pretas" |
| **Interpretabilidade** | Fácil – regras escritas em linguagem próxima do domínio | Depende do modelo (árvores são interpretáveis, deep learning não) |
| **Escalabilidade** | Difícil – número grande de regras gera conflitos e manutenção custosa | Alta – o modelo aprende padrões complexos a partir dos dados |
| **Aprendizado** | Não aprende – requer engenharia manual de regras | Aprende automaticamente a partir de exemplos rotulados |
| **Dados necessários** | Nenhum – basta conhecimento do especialista | Grande volume de dados rotulados |
| **Adaptação** | Manual – toda mudança exige reescrita de regras | Retreino com novos dados pode adaptar o modelo |

**Conclusão:** a abordagem simbólica (regras) é ideal para domínios bem definidos com poucas variáveis e limiares claros, como este caso de conforto térmico. Já o Machine Learning se torna vantajoso quando o número de variáveis cresce, os limites entre classes são difusos, ou quando se dispõe de dados históricos abundantes para treinamento.